In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag, when, avg, lit, abs, round
import numpy as np

jdbc_url = dbutils.secrets.get(scope="Capstone", key="DatabasejdbcUrl")#DatabasejdbcUrl
connection_properties = {
    "user": dbutils.secrets.get(scope="Capstone", key="DatabaseUsername"),
    "password": dbutils.secrets.get(scope="Capstone", key="DatabasePassword"),
    "driver": dbutils.secrets.get(scope="Capstone", key="DatabaseDriver")
}

df_spark_historical_prices_silver = spark.read.parquet(
    "/dbfs/FileStore/Silver/Historical_Prices_Silver.parquet",
    header=True,
    inferSchema=True
)

In [0]:
def calculate_rsi_numpy(close_prices, period=14):
    delta = np.diff(close_prices)
    delta = np.concatenate(([np.nan], delta))

    gains = np.where(delta > 0, delta, 0)
    losses = np.where(delta < 0, -delta, 0)

    avg_gains = np.zeros_like(close_prices)
    avg_losses = np.zeros_like(close_prices)
    rsi = np.zeros_like(close_prices, dtype=float)

    if len(close_prices) >= period:
        avg_gains[period-1] = np.mean(gains[1:period+1])
        avg_losses[period-1] = np.mean(losses[1:period+1])
    
    for i in range(period, len(close_prices)):
        avg_gains[i] = (avg_gains[i-1] * (period - 1) + gains[i]) / period
        avg_losses[i] = (avg_losses[i-1] * (period - 1) + losses[i]) / period
    
    with np.errstate(divide='ignore', invalid='ignore'):
        rs = avg_gains / avg_losses
        rsi = np.where(avg_losses != 0, 100 - (100 / (1 + rs)), 100)
    
    rsi[:period-1] = np.nan
    return np.round(rsi, 2).tolist(), np.round(gains, 2).tolist(), np.round(losses, 2).tolist()

periods = list(range(2, 30))

df_old_data = df_spark_historical_prices_silver.orderBy("Stock_Symbol", "Date").select("Stock_Symbol", "Date", "Close")

for period in periods:
    windowSpec = Window.partitionBy("Stock_Symbol").orderBy("Date")
    df_old_data = df_old_data.withColumn(f"delta_{period}",   \
                        col("Close") - lag("Close", 1)  \
                        .over(windowSpec))

    df_old_data = df_old_data.withColumn(f"gain_{period}",
                        when(col(f"delta_{period}") > 0, col(f"delta_{period}"))    \
                        .otherwise(0))
    
    df_old_data = df_old_data.withColumn(f"loss_{period}", when(col(f"delta_{period}") < 0, -col(f"delta_{period}"))  \
                       .otherwise(0))

    avg_window = Window.partitionBy("Stock_Symbol") \
                        .orderBy("Date")    \
                        .rowsBetween(-period + 1, 0)

    df_old_data = df_old_data.withColumn(f"avg_gain_{period}",    \
                        avg(col(f"gain_{period}"))  \
                        .over(avg_window))  
    
    df_old_data = df_old_data.withColumn(f"avg_loss_{period}",    \
                        avg(col(f"loss_{period}"))  \
                        .over(avg_window))  

    df_old_data = df_old_data.withColumn(f"RS_{period}", when(col(f"avg_loss_{period}") == 0, lit(None))  \
                        .otherwise(col(f"avg_gain_{period}") / col(f"avg_loss_{period}")))
    
    df_old_data = df_old_data.withColumn(f"RSI_{period}",
                        when(col(f"RS_{period}").isNotNull(),   \
                              100 - (100 / (1 + col(f"RS_{period}")))   \
                              ))

selected_cols = ["Stock_Symbol", "Date", "Close"] + \
    [f"RSI_{p}" for p in periods] + \
    [f"gain_{p}" for p in periods] + \
    [f"loss_{p}" for p in periods]

df_result_ta = df_old_data.select(*selected_cols)

for period in periods:
    df_result_ta = df_result_ta.withColumn(f"RSI_{period}", round(col(f"RSI_{period}"), 2))
    df_result_ta = df_result_ta.withColumn(f"gain_{period}", round(col(f"gain_{period}"), 2))
    df_result_ta = df_result_ta.withColumn(f"loss_{period}", round(col(f"loss_{period}"), 2))

for period in periods:
    ma_window = Window.partitionBy("Stock_Symbol").orderBy("Date").rowsBetween(-period + 1, 0)

    df_result_ta = df_result_ta.withColumn(f"rolling_avg_{period}", \
                                    avg(col("Close")).over(ma_window))

    df_result_ta = df_result_ta.withColumn(f"abs_diff_{period}",    \
                                    abs(col("Close") - col(f"rolling_avg_{period}")))

    df_result_ta = df_result_ta.withColumn(f"rel_diff_{period}",    \
                                    when(col("Close") != 0, \
                                         col(f"abs_diff_{period}") / col("Close"))  \
                                    .otherwise(lit(None)))

    df_result_ta = df_result_ta.withColumn(f"MARD_{period}",    \
                                    avg(col(f"rel_diff_{period}")).over(ma_window))
    
    df_result_ta = df_result_ta.withColumn(f"MARD_{period}", round(col(f"MARD_{period}"), 4))
    df_result_ta = df_result_ta.drop(f"rolling_avg_{period}", f"abs_diff_{period}", f"rel_diff_{period}")

df_result_ta = df_result_ta.fillna(-99999)

In [0]:
df_result_ta.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "Silver.Historical_Prices_TA") \
    .option("user", connection_properties["user"]) \
    .option("password", connection_properties["password"]) \
    .option("driver", connection_properties["driver"]) \
    .mode("overwrite") \
    .option("batchsize", 10000) \
    .option("numPartitions", 8) \
    .save()

print("Data successfully written to Azure SQL Database.")